<a href="https://colab.research.google.com/github/Bezawit-cloud/efficient-llm-finetuning/blob/main/notebooks/cloud_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Adaptive Data Selection & Curriculum Learning for Compute-Efficient LLM Fine-Tuning
## Cloud GPU Execution & Experiment Orchestration Notebook

This notebook orchestrates the complete 2×2 experiment matrix and ablations on a Cloud GPU instance (Google Colab, RunPod, Lambda Labs, or Kaggle):

| Exp | Selection | Ordering | Description |
|:---|:---|:---|:---|
| **E1** | 100% (Full) | Random | Baseline (all 49.4k training examples) |
| **E2** | Random 50% | Random | Uniform random subsampling |
| **E3** | Adaptive 50% | Random | Importance-scored selection (Diversity + Complexity + Length) |
| **E4** | Adaptive 50% | Curriculum | **Full Method** (Adaptive Selection + Easy→Hard Curriculum) |
| **E5** | Random 50% | Curriculum | Curriculum on random subset (Isolation test) |
| **Ablation A** | Adaptive 50% | Random | Diversity score only |
| **Ablation B** | Adaptive 50% | Random | Complexity score only |

**Hardware Target:** 1× NVIDIA GPU with $\ge$ 6–16 GB VRAM (e.g., T4, A100, RTX 3090/4090, L4).

### Step 1: Environment & GPU Verification

In [2]:
# Verify GPU hardware
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"BF16 Support: {torch.cuda.is_bf16_supported()}")

/bin/bash: line 1: nvidia-smi: command not found
PyTorch: 2.11.0+cpu
CUDA Available: False


### Step 2: Install Dependencies

In [3]:
# Install repository requirements
%pip install -r requirements.txt -q

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


### Step 3: Run GPU Smoke Test
Verifies CUDA detection, Qwen2.5-0.5B loading, LoRA parameter attachment, 1 step execution, and peak memory logging.

In [ ]:
!python src/gpu_smoke_test.py

### Step 4: Generate Full 52k Importance Scoring Cache
Computes tri-component scores (Diversity + Complexity + Response Length) on GPU (~45–60s) and caches to `data/scored_alpaca.json`.

In [ ]:
import json
from pathlib import Path
from src.utils import load_config, set_seed
from src.data_utils import load_alpaca_dataset
from src.scoring import score_dataset

config = load_config("configs/base_config.yaml")
set_seed(config["seed"])

cache_path = Path("data/scored_alpaca.json")
if not cache_path.exists():
    print("Loading Alpaca dataset for scoring...")
    train_ds, _ = load_alpaca_dataset(config)
    train_examples = [dict(ex) for ex in train_ds]
    print(f"Computing importance scores for {len(train_examples)} examples on GPU...")
    scored = score_dataset(train_examples, config)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, "w") as f:
        json.dump(scored, f)
    print(f"Scores successfully cached to {cache_path} ({len(scored)} items).")
else:
    print(f"Cached scores already exist at {cache_path}.")

### Step 5: Execute Primary Experiment Suite (E1 – E5)
Runs all experiments sequentially with fixed seed `42` and identical LoRA configuration.

In [ ]:
# E1 — Full Baseline (100% data, random order, 3 epochs)
!python src/train_baseline.py --config configs/exp1_baseline.yaml

# E2 — Random 50% (random order, 3 epochs)
!python src/train_baseline.py --config configs/exp2_random50.yaml

# E3 — Adaptive 50% (random order, 3 epochs)
!python src/train_baseline.py --config configs/exp3_adaptive50_random.yaml

# E4 — Full Method: Adaptive 50% + Curriculum (easy->hard, 3 epochs)
!python src/train_baseline.py --config configs/exp4_adaptive50_curriculum.yaml

# E5 — Random 50% + Curriculum (isolation test, 3 epochs)
!python src/train_baseline.py --config configs/exp5_random50_curriculum.yaml

### Step 6: Execute Ablations (A & B)

In [ ]:
# Ablation A — Diversity Only (50% subset, 3 epochs)
!python src/train_baseline.py --config configs/ablation_diversity_only.yaml

# Ablation B — Complexity Only (50% subset, 3 epochs)
!python src/train_baseline.py --config configs/ablation_complexity_only.yaml

### Step 7: Results Compilation & Comparison Table

In [ ]:
import glob
import pandas as pd

result_files = sorted(glob.glob("outputs/**/results.json", recursive=True))
records = []
for f in result_files:
    with open(f) as fp:
        records.append(json.load(fp))

if records:
    df = pd.DataFrame(records)
    cols = ["experiment_id", "experiment_name", "n_train_examples", "data_fraction", "ordering", "eval_loss", "train_loss", "wall_clock_minutes", "peak_gpu_memory_mb"]
    display_cols = [c for c in cols if c in df.columns]
    display(df[display_cols].sort_values("experiment_id"))
else:
    print("No results.json files found yet.")